# data-cleaning_007

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (147).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `unknown` |
| Trading style | `scalping` |
| Timeframe | `daily` |
| Code cells | 15 |
| Detected functions | next_year_prices, append_future_prices, calculate_gas_storage_value_from_df |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/ipynb/Nat_Gas.csv')
data.head()

In [ ]:
import plotly.express as px
import pandas as pd

# Create the plot
fig = px.line(data, x='Dates', y='Prices', title="Prices over Dates")

# Update layout to rotate x-axis tick labels
fig.update_layout(
    xaxis=dict(
        tickangle=24  # Rotate ticks to 24 degrees
    ),
    xaxis_title="Dates",
    yaxis_title="Prices"
)

# Show the plot
fig.show()


In [ ]:
import pandas as pd

# Convert 'Dates' to datetime and set as index
data['Dates'] = pd.to_datetime(data['Dates'])

data['Year'] = data['Dates'].dt.year
data['Month'] = data['Dates'].dt.month

data.tail()

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array(data[data['Month'] == 1]['Year']).reshape(-1, 1)
Y = np.array(data[data['Month'] == 1]['Prices'])

model = LinearRegression().fit(X, Y)

In [ ]:
model.predict(X)

In [ ]:
def next_year_prices(next_year):
    price_list = []
    for i in np.arange(12):
        # Extract the feature and target for the current month
        X = np.array(data[data['Month'] == 1+i]['Year']).reshape(-1, 1)
        Y = np.array(data[data['Month'] == 1+i]['Prices'])

        # Fit the linear regression model
        model = LinearRegression().fit(X, Y)

        # Predict the price for the next year
        price = model.predict([[next_year]])
        price_list.append(price[0])  # Append the scalar value, not array

    return price_list


In [ ]:
price25 = next_year_prices(2025)
np.array(price25)

In [ ]:
# Assuming 'data' is your main DataFrame with columns 'Year', 'Month', 'Prices'
def append_future_prices(data, next_year):
    # Generate predictions for the next year
    future_prices = next_year_prices(next_year)

    # Create a DataFrame for the next year's predictions
    future_data = pd.DataFrame({
        'Year': [next_year] * 12,  # Repeated next_year for each month
        'Month': list(range(1, 13)),  # Months from 1 to 12
        'Prices': future_prices  # Predicted prices
    })

    # Concatenate the original data with the future data
    updated_data = pd.concat([data, future_data], ignore_index=True)

    return updated_data

# Example usage
next_year = 2025
updated_data = append_future_prices(data, next_year)
print(updated_data.tail())


In [ ]:
import pandas as pd

# Convert Year and Month to Datetime, setting day as the last day of the month
updated_data['Datetime'] = pd.to_datetime(updated_data[['Year', 'Month']].assign(Day=1)) + pd.offsets.MonthEnd(0)

print(updated_data.tail())


In [ ]:
updated_data = updated_data.drop(columns=['Dates'], axis = -1)
updated_data

In [ ]:
data = updated_data
data

In [ ]:
import matplotlib.pyplot as plt

plt.plot(data['Datetime'], data['Prices'])
plt.show()

In [ ]:
# Ensure the datetime column is parsed and set as index
data['Datetime'] = pd.to_datetime(data['Datetime'])
data.set_index('Datetime', inplace=True)  # Set 'Dates' column as the index

# Resample the data to daily frequency and interpolate missing values
data = data.resample('D').interpolate('linear')

# Remove Saturdays and Sundays
data = data[~data.index.dayofweek.isin([5, 6])]

# Reset the index to remove DatetimeIndex
data = data.reset_index()

# Display the result
data


In [ ]:
import pandas as pd

def calculate_gas_storage_value_from_df(
    data,
    injection_dates,
    withdrawal_dates,
    injection_rate,
    withdrawal_rate,
    max_volume,
    storage_cost
):
    """
    Calculate the value of the gas storage contract using a DataFrame.

    Parameters:
        data (DataFrame): DataFrame with 'Datetime' and 'Prices' columns.
        injection_dates (list): Dates of gas injection (as strings or datetime objects).
        withdrawal_dates (list): Dates of gas withdrawal (as strings or datetime objects).
        injection_rate (float): Rate of gas injection per day (units/day).
        withdrawal_rate (float): Rate of gas withdrawal per day (units/day).
        max_volume (float): Maximum storage capacity (units).
        storage_cost (float): Cost of storing one unit of gas per day (currency/unit/day).

    Returns:
        float: Total balance (cash) after considering injections, withdrawals, and storage costs.
    """
    # Ensure the 'Datetime' column is in datetime format and sort data
    data = data.sort_values(by='Datetime')

    # Initialize gas in storage and total cash balance
    gas_in_storage = 0  # Current gas in storage
    balance = 0  # Total balance (cash)

    # Iterate through each row in the DataFrame
    for _, row in data.iterrows():
        date = row['Datetime']
        price = row['Prices']

        # Print the current date and price
        print(f"Processing Date: {date}, Price: {price}")

        # Handle gas injection
        if date in pd.to_datetime(injection_dates):
            inject_amount = min(injection_rate, max_volume - gas_in_storage)
            gas_in_storage += inject_amount
            balance -= ((inject_amount * price) + (inject_amount * storage_cost))  # Cost of injection (storage fee)

            # Print the injection details
            print(f"  Injection Date! Injected: {inject_amount} units")
            print(f"  Updated Gas in Storage: {gas_in_storage} units")
            print(f"  Total Balance after Injection: {balance}")

        # Handle gas withdrawal
        if date in pd.to_datetime(withdrawal_dates):
            withdraw_amount = min(withdrawal_rate, gas_in_storage)
            gas_in_storage -= withdraw_amount
            balance += withdraw_amount * price  # Revenue from withdrawal at current price

            # Print the withdrawal details
            print(f"  Withdrawal Date! Withdrew: {withdraw_amount} units")
            print(f"  Updated Gas in Storage: {gas_in_storage} units")
            print(f"  Total Balance after Withdrawal: {balance}")

        # Add the storage cost for the day (only if gas is in storage)
        if gas_in_storage > 0:
            balance -= gas_in_storage * storage_cost  # Daily cost for storing gas

        # Print the storage cost for the day
        print(f"  Storage Cost for the Day: {gas_in_storage * storage_cost}")

        # Print the current status after each day
        print(f"End of Day {date}: Gas in Storage = {gas_in_storage}, Total Balance = {balance}\n")

    return balance


In [ ]:
import pandas as pd

# Define parameters
injection_dates = ["2024-08-30", "2024-09-03"]
withdrawal_dates = ["2025-01-31", "2025-01-30"]
injection_rate = 50  # units/day
withdrawal_rate = 50  # units/day
max_volume = 200  # units
storage_cost = 0.01  # currency/unit/day

# Calculate the contract value
contract_value = calculate_gas_storage_value_from_df(
    data,
    injection_dates,
    withdrawal_dates,
    injection_rate,
    withdrawal_rate,
    max_volume,
    storage_cost
)

print("Contract Value:", contract_value)
